# 17 — Common-Mode Rejection

**Repo:** `decoherence-free-sensing`  
**Notebook role:** noise-rejection layer  
**Previous:** `13_phase_coordinates.ipynb`  
**Next:** `23_lieb_mattis_states.ipynb`

Notebook 13 introduced the coordinate transform

\[
(\phi_A,\phi_B)
\longrightarrow
(\Phi,\varphi).
\]

Notebook 17 explains why that transform matters.

The engineering objective is not merely changing variables. It is separating the coordinate to reject from the coordinate to estimate.

## Engineering statement

Reject shared noise first. Estimate differential phase second.

Common-mode rejection specifies admissible quantum sensing before entanglement is optimized.


## Notebook outputs

Running this notebook creates:

- `results/figures/17_common_mode_rejection_design_rule_v2.png`
- `results/figures/17_dfs_filter_diagram_v2.png`
- `results/figures/17_shared_noise_dominates_node_readout_v2.png`
- `results/figures/17_common_mode_rejection_in_differential_coordinates_v2.png`
- `results/figures/17_rejection_ratio_vs_noise_strength_v2.png`
- `results/common_mode_rejection_summary.csv`
- `results/common_mode_rejection_summary.json`
- `results/decoherence_free_sensing_common_mode_rejection_outputs.zip`


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})


## 1. Coordinate reminder

The two-node phase coordinates are

\[
\Phi=\frac{\phi_A+\phi_B}{2},
\qquad
\varphi=\frac{\phi_A-\phi_B}{2}.
\]

Here:

- \(\Phi\) is shared motion,
- \(\varphi\) is differential information.

In the quantum sensor, the corresponding generators are

\[
J_z^+ = J_z^A+J_z^B,
\qquad
J_z^- = J_z^A-J_z^B.
\]


## 2. Key derivation

Assume the observed phases contain a shared fluctuation plus local readout noise:

\[
\phi_A^{obs}
=
\Phi_{\rm noise}+\varphi+\epsilon_A,
\]

\[
\phi_B^{obs}
=
\Phi_{\rm noise}-\varphi+\epsilon_B.
\]

Then the differential estimate is

\[
\hat{\varphi}
=
\frac{\phi_A^{obs}-\phi_B^{obs}}{2}
=
\varphi+\frac{\epsilon_A-\epsilon_B}{2}.
\]

The common noise \(\Phi_{\rm noise}\) cancels exactly.


## 3. Common-mode rejection design rule

The design rule is:

1. identify the shared phase,
2. constrain the protocol so shared phase is rejected,
3. preserve the differential signal,
4. estimate the differential phase.

The quantum implementation of this rule is the DFS constraint.


In [ ]:
def draw_box(ax, xy, width, height, title, body=None, fontsize=15, facecolor="white", edgecolor="black", lw=1.8):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        linewidth=lw,
        edgecolor=edgecolor,
        facecolor=facecolor
    )
    ax.add_patch(box)
    ax.text(
        x + width/2,
        y + height*0.62 if body else y + height/2,
        title,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold"
    )
    if body:
        ax.text(
            x + width/2,
            y + height*0.28,
            body,
            ha="center",
            va="center",
            fontsize=11
        )

def draw_arrow(ax, p0, p1):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    ))

fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Common-Mode Rejection Design Rule", ha="center", va="center", fontsize=23)

items = [
    (0.05, "Identify", "shared phase\n$\\Phi$"),
    (0.285, "Constrain", "fix DFS sector\n$J_z^+$"),
    (0.52, "Preserve", "differential signal\n$\\varphi$ / $J_z^-$"),
    (0.755, "Estimate", "differential phase\n$\\varphi$"),
]
w, h, y = 0.19, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body)

for x0 in [0.245, 0.48, 0.715]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "Reject shared noise first. Estimate differential phase second.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "17_common_mode_rejection_design_rule_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 4. DFS filter interpretation

A DFS-compatible protocol behaves like a coordinate filter.

The input contains both

\[
\Phi \quad \text{shared noise}
\]

and

\[
\varphi \quad \text{differential signal}.
\]

The DFS constraint fixes \(J_z^+\), making the shared coordinate physically unobservable, while \(J_z^-\) preserves the differential signal.


In [ ]:
fig, ax = plt.subplots(figsize=(12.8, 5.6))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "DFS Filter Diagram", ha="center", va="center", fontsize=23)

draw_box(ax, (0.05, 0.47), 0.21, 0.28, "input phases", r"$\Phi$ shared noise" + "\n" + r"$\varphi$ signal")
draw_box(ax, (0.39, 0.42), 0.22, 0.38, "DFS filter", r"fix $J_z^+$" + "\n" + r"preserve $J_z^-$", facecolor="#eaf3ff", lw=2.3)
draw_box(ax, (0.74, 0.47), 0.21, 0.28, "output", "reject Φ\nestimate φ")

draw_arrow(ax, (0.265, 0.61), (0.38, 0.61))
draw_arrow(ax, (0.615, 0.61), (0.735, 0.61))

ax.text(
    0.5,
    0.22,
    "A DFS-compatible protocol removes the shared coordinate before precision is optimized.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "17_dfs_filter_diagram_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 5. Simulation setup

We simulate repeated phase observations with:

- a true differential phase \(\varphi\),
- a shared noise process \(\Phi_{\rm noise}\),
- independent local noise terms \(\epsilon_A,\epsilon_B\).

Single-node readout follows the shared fluctuation. Differential readout rejects it.


In [ ]:
rng = np.random.default_rng(17)

nshots = 1200
true_varphi = 0.28
sigma_common = 1.15
sigma_local = 0.08

common_noise = rng.normal(0, sigma_common, nshots)
local_a = rng.normal(0, sigma_local, nshots)
local_b = rng.normal(0, sigma_local, nshots)

phi_A_obs = common_noise + true_varphi + local_a
phi_B_obs = common_noise - true_varphi + local_b

common_estimate = (phi_A_obs + phi_B_obs) / 2
differential_estimate = (phi_A_obs - phi_B_obs) / 2

pd.DataFrame({
    "phi_A_obs": phi_A_obs,
    "phi_B_obs": phi_B_obs,
    "common_estimate": common_estimate,
    "differential_estimate": differential_estimate,
}).head()


## 6. Failure mode: individual node readout

Individual node readout faithfully follows common-mode noise. That is the failure mode.

The observed node signals are noisy not because the differential phase is absent, but because the node coordinates are the wrong coordinates for rejecting shared motion.


In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 6.2))

idx = np.arange(nshots)
ax.plot(idx[:220], phi_A_obs[:220], linewidth=1.2, alpha=0.75, label=r"node A readout $\phi_A^{obs}$")
ax.plot(idx[:220], phi_B_obs[:220], linewidth=1.2, alpha=0.75, label=r"node B readout $\phi_B^{obs}$")
ax.axhline(true_varphi, linestyle="--", linewidth=1.4, label=r"true $+\varphi$")
ax.axhline(-true_varphi, linestyle=":", linewidth=1.4, label=r"true $-\varphi$")

ax.set_title("Shared Noise Dominates Individual Node Readout")
ax.set_xlabel("shot index")
ax.set_ylabel("observed phase")
ax.grid(True, alpha=0.28)
ax.legend(loc="upper right")

ax.text(
    8,
    2.6,
    "shared noise dominates\nsingle-node readout",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "17_shared_noise_dominates_node_readout_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 7. Differential coordinates reject the common mode

The common coordinate remains broad because it contains the shared noise.

The differential coordinate remains concentrated near the true \(\varphi\) because the shared term cancels.


In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 6.1))

ax.hist(common_estimate, bins=45, alpha=0.62, label=r"common coordinate $\hat{\Phi}$")
ax.hist(differential_estimate, bins=45, alpha=0.88, label=r"differential estimate $\hat{\varphi}$")
ax.axvline(true_varphi, linestyle="--", linewidth=1.8, label=r"true $\varphi$")

ax.set_title("Common-Mode Rejection in Differential Coordinates")
ax.set_xlabel("estimate value")
ax.set_ylabel("count")
ax.grid(True, alpha=0.25)
ax.legend()

path = FIGURES / "17_common_mode_rejection_in_differential_coordinates_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 8. Rejection ratio versus common-noise strength

As common-mode noise increases, a single-node estimate degrades.

The differential estimate remains set primarily by local differential noise because the shared component cancels.

We quantify this with the variance ratio:

\[
R =
\frac{\mathrm{Var}(\phi_A^{obs})}{\mathrm{Var}(\hat{\varphi})}.
\]

A larger \(R\) means stronger common-mode rejection.


In [ ]:
sigma_common_values = np.logspace(-2, 1.1, 42)
nmc = 5000
ratios = []
node_vars = []
diff_vars = []

for sc in sigma_common_values:
    eta = rng.normal(0, sc, nmc)
    ea = rng.normal(0, sigma_local, nmc)
    eb = rng.normal(0, sigma_local, nmc)

    a = eta + true_varphi + ea
    b = eta - true_varphi + eb
    diff = (a - b) / 2

    node_var = np.var(a)
    diff_var = np.var(diff)

    node_vars.append(node_var)
    diff_vars.append(diff_var)
    ratios.append(node_var / diff_var)

fig, ax = plt.subplots(figsize=(8.2, 6.0))

ax.loglog(sigma_common_values, node_vars, linewidth=2.0, label="single-node variance")
ax.loglog(sigma_common_values, diff_vars, linewidth=2.0, label="differential variance")
ax.loglog(sigma_common_values, ratios, linestyle="--", linewidth=2.0, label="rejection ratio")

ax.set_title("Common-Mode Rejection Improves as Shared Noise Grows")
ax.set_xlabel(r"common-noise strength $\sigma_\Phi$")
ax.set_ylabel("variance / ratio")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

path = FIGURES / "17_rejection_ratio_vs_noise_strength_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 9. Engineering checklist

A common-mode-rejecting sensor should satisfy:

1. identify the shared coordinate,
2. reject \(\Phi\) through the DFS constraint,
3. preserve \(\varphi\),
4. estimate only the differential phase,
5. optimize entanglement only after the admissible coordinate structure is fixed.


## 10. Summary table


In [ ]:
summary = pd.DataFrame([
    {
        "item": "observed_node_A",
        "value": "phi_A_obs = Phi_noise + varphi + epsilon_A",
        "engineering_role": "single-node readout contains shared noise"
    },
    {
        "item": "observed_node_B",
        "value": "phi_B_obs = Phi_noise - varphi + epsilon_B",
        "engineering_role": "single-node readout contains shared noise"
    },
    {
        "item": "differential_estimate",
        "value": "varphi_hat = (phi_A_obs - phi_B_obs) / 2",
        "engineering_role": "cancels shared phase"
    },
    {
        "item": "cancelled_term",
        "value": "Phi_noise",
        "engineering_role": "dominant common-mode fluctuation removed"
    },
    {
        "item": "remaining_noise",
        "value": "(epsilon_A - epsilon_B) / 2",
        "engineering_role": "local differential noise remains"
    },
    {
        "item": "dfs_common_generator",
        "value": "J_z^+",
        "engineering_role": "quantum constraint corresponding to common coordinate"
    },
    {
        "item": "dfs_signal_generator",
        "value": "J_z^-",
        "engineering_role": "quantum generator corresponding to differential coordinate"
    },
    {
        "item": "design_rule",
        "value": "Reject shared noise first; estimate differential phase second",
        "engineering_role": "system-level specification"
    },
])

summary_csv = RESULTS / "common_mode_rejection_summary.csv"
summary_json = RESULTS / "common_mode_rejection_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 11. Export notebook outputs

This cell creates one zip archive containing all generated figures and summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_common_mode_rejection_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(FIGURES.glob("17_*_v2.png")):
        zf.write(file, file.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created: {zip_path}")

# Optional Colab download:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

zip_path


## 12. Next notebook

Suggested next notebook:

`23_lieb_mattis_states.ipynb`

Core goal:

- construct the DFS-compatible Lieb-Mattis target state,
- show how the state lives inside the central DFS sector,
- connect singlet structure to differential phase sensitivity,
- prepare the transition from noise rejection to entanglement advantage.
